# Task 2.3 - Hyperparameter Experimentation
**Skill Set Go EduTech - AI/ML Internship - Week 2**

**Goal:** run controlled experiments varying optimizer, learning rate, batch size and epochs, and compare the outcomes.

**Rule I follow:** dataset, architecture and evaluation procedure stay fixed. Starting from one baseline run (R01),
each new run changes **only one thing**, so any difference in the result can be blamed on that change.

To keep 12 runs fast, I use a fixed subset of CIFAR-10 (10,000 training / 2,000 validation images) and the same
small CNN as Task 2.2.

In [ ]:
import os, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

import torchvision
import torchvision.transforms as T

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
os.makedirs("results", exist_ok=True)

## 1. Fixed data (same for every run)

In [ ]:
MEAN = (0.4914, 0.4822, 0.4465)
STD  = (0.2470, 0.2435, 0.2616)
tf = T.Compose([T.ToTensor(), T.Normalize(MEAN, STD)])       # no random augmentation -> identical data every run
full = torchvision.datasets.CIFAR10("./data", train=True, download=True, transform=tf)

g = torch.Generator().manual_seed(42)
perm = torch.randperm(50000, generator=g).tolist()
train_idx, val_idx = perm[:10000], perm[10000:12000]

def to_tensors(indices):
    xs, ys = [], []
    for i in indices:
        x, y = full[i]
        xs.append(x); ys.append(y)
    return torch.stack(xs), torch.tensor(ys)

X_train, y_train = to_tensors(train_idx)
X_val, y_val = to_tensors(val_idx)
train_set = TensorDataset(X_train, y_train)
val_set = TensorDataset(X_val, y_val)
print("Train:", X_train.shape, "| Val:", X_val.shape)

## 2. Fixed model and the experiment function

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3),
            nn.Linear(128 * 4 * 4, 128), nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


def run_experiment(run_id, opt_name, lr, batch_size, epochs, seed=42):
    torch.manual_seed(seed)                       # same starting weights and batch order logic for every run
    model = SmallCNN().to(device)
    criterion = nn.CrossEntropyLoss()

    if opt_name == "SGD":
        optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    elif opt_name == "SGD+momentum":
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    elif opt_name == "Adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    else:
        raise ValueError("unknown optimizer " + opt_name)

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=500, shuffle=False)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    start = time.time()
    for epoch in range(epochs):
        model.train()
        loss_sum, correct, seen = 0.0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * xb.size(0)
            correct += (out.argmax(1) == yb).sum().item()
            seen += xb.size(0)

        model.eval()
        v_loss, v_correct = 0.0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                out = model(xb)
                v_loss += criterion(out, yb).item() * xb.size(0)
                v_correct += (out.argmax(1) == yb).sum().item()

        history["train_loss"].append(loss_sum / seen)
        history["train_acc"].append(correct / seen)
        history["val_loss"].append(v_loss / len(val_set))
        history["val_acc"].append(v_correct / len(val_set))

    best_epoch = int(np.argmax(history["val_acc"])) + 1
    row = {
        "run_id": run_id, "optimizer": opt_name, "lr": lr, "batch_size": batch_size, "epochs": epochs,
        "final_train_acc": round(history["train_acc"][-1], 4),
        "final_val_acc": round(history["val_acc"][-1], 4),
        "final_val_loss": round(history["val_loss"][-1], 4),
        "best_val_acc": round(max(history["val_acc"]), 4),
        "best_epoch": best_epoch,
        "time_sec": round(time.time() - start, 1),
    }
    return row, history

## 3. Experiment plan
Baseline **R01**: Adam, lr = 0.001, batch size 64, 5 epochs. Every other run changes one setting compared with R01.

| Group | Runs | What changes |
|---|---|---|
| Baseline | R01 | - |
| Learning rate | R02, R03, R04 | lr = 0.1, 0.01, 0.0001 |
| Optimizer | R05, R06 | SGD (lr 0.01), SGD + momentum (lr 0.01) |
| Batch size | R07, R08, R09 | 32, 128, 256 |
| Epochs | R10 | 20 epochs (to look for under/overfitting) |

In [ ]:
plan = [
    # run_id, optimizer, lr, batch_size, epochs
    ("R01", "Adam", 0.001, 64, 5),      # baseline
    ("R02", "Adam", 0.1, 64, 5),        # learning rate
    ("R03", "Adam", 0.01, 64, 5),
    ("R04", "Adam", 0.0001, 64, 5),
    ("R05", "SGD", 0.01, 64, 5),        # optimizer
    ("R06", "SGD+momentum", 0.01, 64, 5),
    ("R07", "Adam", 0.001, 32, 5),      # batch size
    ("R08", "Adam", 0.001, 128, 5),
    ("R09", "Adam", 0.001, 256, 5),
    ("R10", "Adam", 0.001, 64, 20),     # epochs
]

rows, histories = [], {}
for run_id, opt, lr, bs, ep in plan:
    row, hist = run_experiment(run_id, opt, lr, bs, ep)
    rows.append(row)
    histories[run_id] = hist
    print(f"{run_id} | {opt:12s} lr={lr:<7} bs={bs:<3} ep={ep:<2} -> train acc {row['final_train_acc']:.3f}, "
          f"val acc {row['final_val_acc']:.3f}, val loss {row['final_val_loss']:.3f} ({row['time_sec']}s)")

## 4. Comparison table

In [ ]:
df = pd.DataFrame(rows)
df["train_val_gap"] = (df["final_train_acc"] - df["final_val_acc"]).round(4)
df

## 5. Charts
Both charts have labelled axes and titles.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#888888" if r == "R01" else "#2f7d3c" for r in df["run_id"]]
ax.bar(df["run_id"], df["final_val_acc"], color=colors)
ax.set_xlabel("Run ID (R01 = baseline, grey)")
ax.set_ylabel("Final validation accuracy")
ax.set_title("Validation accuracy by experiment run")
ax.set_ylim(0, 1)
for i, v in enumerate(df["final_val_acc"]):
    ax.text(i, v + 0.01, f"{v:.2f}", ha="center", fontsize=8)
plt.tight_layout()
plt.savefig("results/val_acc_by_run.png", dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
h = histories["R10"]
ax[0].plot(range(1, 21), h["train_acc"], label="Train")
ax[0].plot(range(1, 21), h["val_acc"], label="Validation")
ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Accuracy"); ax[0].set_title("R10: 20 epochs, accuracy"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(range(1, 21), h["train_loss"], label="Train")
ax[1].plot(range(1, 21), h["val_loss"], label="Validation")
ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Loss"); ax[1].set_title("R10: 20 epochs, loss"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig("results/epochs_curve_R10.png", dpi=150)
plt.show()

## 6. My observations per run
*(Fill this in yourself after looking at the table and charts. For each run write what happened - for example
"training accuracy stuck near 10%: lr too high", "slower but stable", "validation loss started rising after epoch X".)*

In [ ]:
notes = {
    "R01": "",
    "R02": "",
    "R03": "",
    "R04": "",
    "R05": "",
    "R06": "",
    "R07": "",
    "R08": "",
    "R09": "",
    "R10": "",
}
df["notes"] = df["run_id"].map(notes)
df.to_csv("results/experiment_log.csv", index=False)
with open("results/histories.json", "w") as f:
    json.dump(histories, f)
df

## 7. Choosing the final configuration
I should not just pick the highest number from one run - it could be luck. So I decide from the whole table, look at the
validation behaviour (not only training accuracy), and then **re-run the chosen setting with 2 more seeds** to see whether
the result is stable.

Set `FINAL_RUN` below to the run ID you choose after reviewing the table.

In [ ]:
FINAL_RUN = None      # <- put your chosen run ID here, e.g. "R01"

assert FINAL_RUN is not None, "Choose FINAL_RUN after reviewing the table"
cfg = df[df["run_id"] == FINAL_RUN].iloc[0]
seed_accs = [float(cfg["final_val_acc"])]
for s in (1, 2):
    row, _ = run_experiment(FINAL_RUN + f"-seed{s}", cfg["optimizer"], float(cfg["lr"]),
                            int(cfg["batch_size"]), int(cfg["epochs"]), seed=s)
    seed_accs.append(row["final_val_acc"])

print("Validation accuracy over 3 seeds:", seed_accs)
print(f"Mean {np.mean(seed_accs):.4f} | Std {np.std(seed_accs):.4f}")

final = {
    "final_run": FINAL_RUN, "optimizer": cfg["optimizer"], "lr": float(cfg["lr"]),
    "batch_size": int(cfg["batch_size"]), "epochs": int(cfg["epochs"]),
    "val_acc_seeds": seed_accs, "val_acc_mean": round(float(np.mean(seed_accs)), 4),
    "val_acc_std": round(float(np.std(seed_accs)), 4),
}
with open("results/final_config.json", "w") as f:
    json.dump(final, f, indent=2)
final

## 8. Why I selected this configuration
*(Write in your own words: what did the table show, what changed, what happened, and why did you pick this run -
mention validation behaviour and the seed check.)*

**My reasoning:**

-